# 📊 EDA & Exploration: 014_segformer_polyp

Self-contained sketchpad for experimenting with **014_segformer_polyp**.

**Goal:** Understand data distribution, visualize characteristics, prototype the architecture, and run a fast lightweight training loop before creating a formal experiment.

In [ ]:
# === 1. Environment Setup ===
try:
    from google.colab import drive
    drive.mount('/content/drive')  # For data access
    IN_COLAB = True
    # Clone code from GitHub (deterministic, no Drive sync delay)
    !git clone --depth 1 -q https://github.com/khangnghiem/deep-learning.git /content/deep-learning 2>/dev/null || true
except ImportError:
    IN_COLAB = False

import sys
if IN_COLAB:
    sys.path.insert(0, '/content/deep-learning')
else:
    sys.path.insert(0, '..')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm.auto import tqdm

# Use seaborn styles for better looking dashboards
sns.set_theme(style="whitegrid")

# --- Standalone Environment Configuration ---
import os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
    DRIVE_ROOT = Path('/content/drive/MyDrive')
except ImportError:
    IN_COLAB = False
    DRIVE_ROOT = Path(os.environ.get('DRIVE_ROOT', 'G:/My Drive'))

DATA_LAKE = DRIVE_ROOT / 'data_lake'
BRONZE = DATA_LAKE / '01_bronze_medical'
SILVER = DATA_LAKE / '02_silver'
GOLD = DATA_LAKE / '03_gold'
MODELS = DRIVE_ROOT / 'models'
PRETRAINED = MODELS / 'pretrained'
TRAINED = MODELS / 'trained'
OPS = DRIVE_ROOT / 'ops'
MLFLOW_TRACKING_URI = f'file://{OPS / "mlflow" / "mlruns"}'
REPOS = DRIVE_ROOT / 'repos'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load Data

In [ ]:
from torchvision import datasets, transforms

# TODO: Implement specific transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
]) # Add target augmentations here

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# TODO: Load your dataset (from BRONZE / SILVER / GOLD directory)
data_path = BRONZE / "014_segformer_polyp"

# Example placeholders. Replace with actual Dataset instantiation.
# train_dataset = datasets.ImageFolder(data_path / "train", transform=train_transform)
# val_dataset = datasets.ImageFolder(data_path / "val", transform=val_transform)

# For notebook testing purposes, we define dummy variables
train_dataset = None
val_dataset = None
train_loader = None
val_loader = None

## 3. 📊 Data Distribution Dashboard

In [ ]:
def render_data_dashboard(train_ds, val_ds, num_samples=8):
    """Render an inline dashboard of data characteristics."""
    if train_ds is None or val_ds is None:
        print("Datasets not loaded yet.")
        return
        
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 2)
    
    # --- 1. Class Distribution ---
    ax_dist = fig.add_subplot(gs[0, 0])
    if hasattr(train_ds, 'targets'):
        train_targets = train_ds.targets
    else:
        train_targets = [y for _, y in train_ds]  # Note: Can be slow for large datasets
    if train_targets:
        sns.countplot(x=train_targets, ax=ax_dist, palette="viridis")
        ax_dist.set_title("Train Class Distribution", fontsize=14)
        ax_dist.set_xlabel("Class ID")
        ax_dist.set_ylabel("Count")
    
    # --- 2. Train/Val Split Stats ---
    ax_split = fig.add_subplot(gs[0, 1])
    labels = ['Train', 'Validation']
    sizes = [len(train_ds), len(val_ds)]
    ax_split.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=['#4CAF50', '#2196F3'])
    ax_split.set_title(f"Split Ratio (Total: {sum(sizes)} samples)", fontsize=14)
    
    # --- 3. Sample Grid ---
    fig_grid, axes = plt.subplots(2, 4, figsize=(16, 6))
    fig_grid.suptitle("Sample Images (Augmented)", fontsize=14)
    sample_indices = np.random.choice(len(train_ds), min(num_samples, len(train_ds)), replace=False)
    for idx, sample_idx in enumerate(sample_indices):
        ax = axes[idx // 4, idx % 4]
        x, y = train_ds[sample_idx]
        if isinstance(x, torch.Tensor):
            if x.shape[0] == 3:
               img = x.permute(1, 2, 0).numpy()
               img = np.clip(img, 0, 1)
            else:
                img = x.squeeze().numpy()
        else:
            img = x
        ax.imshow(img, cmap='gray' if len(img.shape)==2 else None)
        ax.set_title(f"Class: {y}", fontsize=10)
        ax.axis('off')
    for idx in range(len(sample_indices), 8):
        axes[idx // 4, idx % 4].axis('off')

    plt.tight_layout()
    plt.show()

# render_data_dashboard(train_dataset, val_dataset)

## 4. Model Architecture Prototype

In [ ]:
class PrototypeModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # TODO: Define architecture
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(224*224*3, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

model = PrototypeModel(num_classes=10).to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Prototype model instantiated. Trainable params: {num_params:,}")

## 5. Lightweight Training Loop
No MLflow tracking, fast epochs, just checking if it learns.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 3  # Keep it short for prototyping

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

# Ensure data is loaded before running
def run_training():
    if train_loader is None:
        print("Dataloaders not initialized.")
        return

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
        
        history['train_loss'].append(running_loss / len(train_loader))
        history['train_acc'].append(correct / total)

        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]  "):
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)
                running_loss += loss.item()
                correct += (out.argmax(1) == y).sum().item()
                total += y.size(0)
                
        history['val_loss'].append(running_loss / len(val_loader))
        history['val_acc'].append(correct / total)
        
        print(f"Epoch {epoch+1}: Train Acc {history['train_acc'][-1]:.4f} | Val Acc {history['val_acc'][-1]:.4f}")

# run_training()

## 6. 📈 Results Dashboard

In [ ]:
from sklearn.metrics import confusion_matrix

def render_results_dashboard(history, model, val_loader, class_names=None):
    """Render an inline dashboard of model performance."""
    if not history['train_loss']:
        print("No training history available.")
        return
        
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(2, 2)

    # --- 1. Loss & Accuracy Curves ---
    ax_loss = fig.add_subplot(gs[0, 0])
    ax_loss.plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
    ax_loss.plot(history['val_loss'], label='Val Loss', marker='o', linewidth=2)
    ax_loss.set_title("Learning Curves (Loss)", fontsize=14)
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend()

    ax_acc = fig.add_subplot(gs[0, 1])
    ax_acc.plot(history['train_acc'], label='Train Acc', marker='o', linewidth=2, color='green')
    ax_acc.plot(history['val_acc'], label='Val Acc', marker='o', linewidth=2, color='orange')
    ax_acc.set_title("Learning Curves (Accuracy)", fontsize=14)
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.legend()

    # --- 2. Confusion Matrix ---
    if val_loader:
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                out = model(x)
                all_preds.extend(out.argmax(1).cpu().numpy())
                all_labels.extend(y.numpy())
        
        cm = confusion_matrix(all_labels, all_preds)
        ax_cm = fig.add_subplot(gs[1, :])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm, 
                    xticklabels=class_names, yticklabels=class_names)
        ax_cm.set_title("Confusion Matrix", fontsize=14)
        ax_cm.set_xlabel("Predicted Label")
        ax_cm.set_ylabel("True Label")

    plt.tight_layout()
    plt.show()

# class_names = [str(i) for i in range(10)]
# render_results_dashboard(history, model, val_loader, class_names)

## 7. Observations & Next Steps

**Observations:**
- ...

**Suggested Hyperparameters for `config.yaml`:**
- Learning Rate: ...
- Batch Size: ...
- Augmentations: ...